# Chapter 4 Notebook

## Setup

In [0]:
%pip install -r ../requirements.txt
dbutils.library.restartPython()

## Databricks LangChain Components

In [0]:
from databricks_langchain import ChatDatabricks

chat_model = ChatDatabricks(
    endpoint="databricks-gpt-oss-120b",
    temperature=0,
    max_tokens=256,
)

#invoke the chat model
print(chat_model.invoke("How do I book flights with Unity Airways?"))

In [0]:
from databricks_langchain import DatabricksVectorSearch

index_name = "workspace.unity_airways.faq_index"
endpoint_name = "vs_endpoint"

vector_store = DatabricksVectorSearch(
    endpoint=endpoint_name,
    index_name=index_name,
)

results = vector_store.similarity_search(
    query="How do i book flights with Unity Airways?", k=1
)
for doc in results:
    print(f"* {doc.page_content} [{doc.metadata}]")

In [0]:
retriever = vector_store.as_retriever(search_kwargs= {'k': 1, "query_type": "ANN"})
print(retriever.invoke("How do I book flights with Unity Airways?"))

## Building RAG Chain

### LLM-Only

In [0]:
import mlflow

mlflow.langchain.autolog()

# Set the active model context
logged_model_name = "llm_only"
active_model_info = mlflow.set_active_model(name=logged_model_name)

print(
    f"Active LoggedModel: '{active_model_info.name}', Model ID: '{active_model_info.model_id}'"
)

In [0]:
from databricks_langchain import ChatDatabricks

chat_model = ChatDatabricks(
    endpoint="databricks-gpt-oss-120b",
    temperature=0,
    max_tokens=256
)

query = "How do I book flights with Unity Airways?"
print(chat_model.invoke(query).content)

### Full RAG Chain

In [0]:
import mlflow
mlflow.langchain.autolog()

In [0]:
# Set the active model context
logged_model_name = "rag_chain"
active_model_info = mlflow.set_active_model(name=logged_model_name)

print(
    f"Active LoggedModel: '{active_model_info.name}', Model ID: '{active_model_info.model_id}'"
)

In [0]:
from mlflow.models import ModelConfig

model_config = ModelConfig(development_config="../conf/chapter04_conf.yml")
databricks_resources = model_config.get("databricks_resources")
retriever_config = model_config.get("retriever_config")
llm_config = model_config.get("llm_config")

In [0]:
from databricks_langchain import ChatDatabricks

model = ChatDatabricks(
    endpoint=databricks_resources.get("model_name"),
    **llm_config.get("llm_parameters")
)

In [0]:
from databricks_langchain import DatabricksVectorSearch

vector_search_schema = retriever_config.get("retriever_schema")

vector_search_as_retriever = DatabricksVectorSearch(
    index_name=retriever_config.get("index_name"),
    columns=[
        vector_search_schema.get("primary_key"),
        vector_search_schema.get("chunk_text"),
        vector_search_schema.get("document_uri"),
    ],
).as_retriever(search_kwargs=retriever_config.get("parameters"))

In [0]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template=llm_config.get("llm_prompt_template"),
    input_variables=llm_config.get("llm_prompt_template_variables"),
)

In [0]:
from operator import itemgetter

from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser

from helpers.format_helpers import (
    extract_user_query_string,
    format_docs
)

chain = (
    {
        "context": itemgetter("messages") | RunnableLambda(extract_user_query_string) | vector_search_as_retriever | RunnableLambda(format_docs),
        "question": itemgetter("messages") | RunnableLambda(extract_user_query_string)
    }
    | prompt
    | model
    | StrOutputParser()
)

In [0]:
from PIL import Image
from io import BytesIO
from IPython.display import display as image_display

image_byte = chain.get_graph().draw_mermaid_png()
image = Image.open(BytesIO(image_byte))
image_display(image)

image.save("chain_image.png", "PNG")

In [0]:
chain.invoke({"messages": [{"role": "user", "content": "How do I book flights with Unity Airways?"}]})

In [0]:
app_params = {
    "model_name": model_config.get("databricks_resources").get("model_name"),
    "retriever_config": model_config.get("retriever_config").get("parameters"),
    "llm_config": model_config.get("llm_config").get("llm_parameters"),
}
mlflow.log_model_params(model_id=active_model_info.model_id, params=app_params)

## Logging LangChain RAG in MLflow

In [0]:
import os
from mlflow.models import ModelConfig
from mlflow.models import infer_signature
from mlflow.models.resources import (DatabricksServingEndpoint, DatabricksVectorSearchIndex)

In [0]:
# Set the active model context
logged_model_name = "rag_chain_with_artifacts"
active_model_info = mlflow.set_active_model(name=logged_model_name)

print(
    f"Active LoggedModel: '{active_model_info.name}', Model ID: '{active_model_info.model_id}'"
)

In [0]:
rag_chain_conf_path = "../conf/chapter04_conf.yml"
model_config = ModelConfig(development_config=rag_chain_conf_path)

In [0]:
lc_model_signature = infer_signature(
    model_input=model_config.get("input_example"),
    model_output=model_config.get("output_example"),
)

lc_model_signature

In [0]:
from mlflow.models.resources import (DatabricksServingEndpoint, DatabricksVectorSearchIndex)

dependent_resources = [
  DatabricksServingEndpoint(endpoint_name=databricks_resources.get("model_name")),DatabricksVectorSearchIndex(index_name=retriever_config.get("index_name"))
  ]

In [0]:
def load_retriever(persist_directory):
  vector_search_schema = retriever_config.get("retriever_schema")
  vector_search_as_retriever = DatabricksVectorSearch(
      index_name=retriever_config.get("index_name"),
      columns=[
          vector_search_schema.get("primary_key"),
          vector_search_schema.get("chunk_text"),
          vector_search_schema.get("document_uri"),
      ],
  ).as_retriever(search_kwargs=retriever_config.get("parameters"))
  return vector_search_as_retriever

In [0]:
image_filename = "chain_image.png"

with mlflow.start_run():

    logged_chain_info = mlflow.langchain.log_model(
        lc_model=chain,
        input_example=model_config.get("input_example"),
        signature=lc_model_signature,
        resources=dependent_resources,
        pip_requirements="../requirements.txt",
        loader_fn = load_retriever,
        name = active_model_info.name,
        model_id = active_model_info.model_id)

    mlflow.log_image(mlflow.Image(os.path.join(os.getcwd(), image_filename)), image_filename)


In [0]:
# Test the chain locally
loaded_chain = mlflow.langchain.load_model(logged_chain_info.model_uri)
loaded_chain.invoke(model_config.get("input_example"))

## Models from Code

### Create new rag_chain.py script

In [0]:
%%writefile rag_chain.py

import os
import mlflow

from operator import itemgetter

from databricks_langchain import ChatDatabricks
from databricks_langchain import DatabricksVectorSearch

from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables import ConfigurableField

from helpers.format_helpers import (
    extract_user_query_string,
    extract_previous_messages,
    combine_all_messages_for_vector_search,
    format_context,
    format_docs
)

## Enable MLflow Tracing
mlflow.langchain.autolog()

## Get the conf from the local conf file
model_config = mlflow.models.ModelConfig(development_config="../conf/chapter04_conf.yml")
databricks_resources = model_config.get("databricks_resources")
retriever_config = model_config.get("retriever_config")
llm_config = model_config.get("llm_config")

## Vector Search
vector_search_schema = retriever_config.get("retriever_schema")

vector_search_as_retriever = DatabricksVectorSearch(
    index_name=retriever_config.get("index_name"),
    columns=[
        vector_search_schema.get("primary_key"),
        vector_search_schema.get("chunk_text"),
        vector_search_schema.get("document_uri"),
    ],
).as_retriever(search_kwargs=retriever_config.get("parameters"))


mlflow.models.set_retriever_schema(
    primary_key=vector_search_schema.get("primary_key"),
    text_column=vector_search_schema.get("chunk_text"),
    doc_uri=vector_search_schema.get("document_uri"),
)

## Prompt Template
prompt = PromptTemplate(
    template=llm_config.get("llm_prompt_template"),
    input_variables=llm_config.get("llm_prompt_template_variables"),
)

## LLM
model = ChatDatabricks(
    endpoint=databricks_resources.get("model_name"),
    **llm_config.get("llm_parameters")
)

## RAG Chain
chain = (
    {
        "context": itemgetter("messages") | RunnableLambda(extract_user_query_string) | vector_search_as_retriever | RunnableLambda(format_docs),
        "question": itemgetter("messages") | RunnableLambda(extract_user_query_string)
    }
    | prompt
    | model
    | StrOutputParser()
)

## Set Model for Models from Code Logging to Work
mlflow.models.set_model(model=chain)

In [0]:
dbutils.library.restartPython()

In [0]:
from mlflow.models import ModelConfig

model_config = ModelConfig(development_config="../conf/chapter04_conf.yml")
databricks_resources = model_config.get("databricks_resources")
retriever_config = model_config.get("retriever_config")
llm_config = model_config.get("llm_config")

from rag_chain import chain

In [0]:
chain.invoke(model_config.get("input_example"))

### Log to MLflow

In [0]:
import mlflow 

# Set the active model context
logged_model_name = "rag_chain_models_from_code"
active_model_info = mlflow.set_active_model(name=logged_model_name)

print(
    f"Active LoggedModel: '{active_model_info.name}', Model ID: '{active_model_info.model_id}'"
)

In [0]:
import os
from mlflow.models import ModelConfig
from mlflow.models import infer_signature
from mlflow.models.resources import (DatabricksServingEndpoint, DatabricksVectorSearchIndex)

lc_model_signature = infer_signature(
    model_input=model_config.get("input_example"),
    model_output=model_config.get("output_example"),
)

dependent_resources = [
  DatabricksServingEndpoint(endpoint_name=databricks_resources.get("model_name")),DatabricksVectorSearchIndex(index_name=retriever_config.get("index_name"))
  ]

In [0]:
image_filename = "chain_image.png"
rag_chain_script_path = "rag_chain.py"
helpers_path = ["helpers/"]
rag_chain_conf_path = "../conf/chapter04_conf.yml"

uc_model_conf_path = "../conf/uc_model_registry.yml"
uc_model_conf = ModelConfig(development_config=uc_model_conf_path)

with mlflow.start_run():

    logged_chain_info = mlflow.langchain.log_model(
        lc_model=rag_chain_script_path,
        input_example=model_config.get("input_example"),
        signature=lc_model_signature,
        resources=dependent_resources,
        pip_requirements="../requirements.txt",
        code_paths=helpers_path,
        model_config=rag_chain_conf_path,
        name=active_model_info.name,
        model_id=active_model_info.model_id,
        registered_model_name=uc_model_conf.get("rag_chain_model").get("full_name"),
    )

    mlflow.log_image(
        mlflow.Image(os.path.join(os.getcwd(), image_filename)), image_filename
    )

In [0]:
# Test the chain locally
loaded_chain = mlflow.langchain.load_model(logged_chain_info.model_uri)
loaded_chain.invoke(model_config.get("input_example"))